In [1]:
import pandas as pd

base_dados = [
    "../data/raw/1semestre/Base de dados paradas dash de departamento - Geral 2026 .01.xlsx",
    "../data/raw/base_atual/Base de Dados Paradas.xlsx",
]

partes = [pd.read_excel(a, sheet_name="PARADAS") for a in base_dados]
par = pd.concat(partes, ignore_index=True)

for coluna in ["Data inicial","Data final"]:
    par[coluna] = pd.to_datetime(par[coluna], format="mixed",dayfirst=True)

par["dur_min"] = pd.to_timedelta(par["Duração"].astype(str)).dt.total_seconds() / 60
par = par.drop(columns=["Duração", "Operador"])

par.to_parquet("../data/processed/paradas_bruto.parquet")

In [ ]:
import pandas as pd

base_dados = [
    "../data/raw/1semestre/Base de dados paradas dash de departamento - Geral 2026 .01.xlsx",
    "../data/raw/base_atual/Base de Dados Paradas.xlsx",
]

partes = [pd.read_excel(a, sheet_name="PARADAS") for a in base_dados]
par = pd.concat(partes, ignore_index=True)

print(par["Data inicial"].map(type).value_counts())

As colunas de data vêm misturando data e texto. Foi necessário converter antes de usar.

In [ ]:
import pandas as pd
par = pd.read_parquet("../data/processed/paradas_bruto.parquet")

print("Quantas linhas e colunas?")
print("linhas: ", par.shape[0],"\nColunas: ", par.shape[1])
print("\n")
print("Tipo de cada coluna:")
par.info()
print("\n")
print("Período coberto:")
print(par["Data inicial"].min(), "até", par["Data inicial"].max())
print("\n")
print("Quantas máquinas (estações) diferentes?")
print("Estações diferentes:", par["Estação"].nunique())
print("\n")
print("Quantos eventos têm a MC 68 e a MC 63?")
mc68 = par[par["Estação"].str.startswith("MC 68")]
mc63 = par[par["Estação"].str.startswith("MC 63")]
print("MC 68:", len(mc68), "MC 63:", len(mc63)) ## maquinas modelo escolhidas para estudo inicial
print("\n")
print("Horas paradas por tipo de evento na MC 68 (top 10):")
top68 = mc68.groupby("Evento")["dur_min"].sum().sort_values(ascending=False) / 60
print(top68.head(10))
print("\n")
print("Horas paradas por tipo de evento na MC 63 (top 10):")
top63 = mc63.groupby("Evento")["dur_min"].sum().sort_values(ascending=False) / 60
print(top63.head(10))
print("\n")

print("Que % do tempo parado é \"Parada não Categorizada\"?")
total = mc68["dur_min"].sum()
nao_cat = mc68[mc68["Evento"].str.contains("não Categorizada", case=False)]["dur_min"].sum()
print(f"MC 68: {100 * nao_cat / total:.1f}%")
print(f"MC 63: {100 * mc63[mc63['Evento'].str.contains('não Categorizada', case=False)]['dur_min'].sum() / mc63['dur_min'].sum():.1f}%")

print("\n")
print("Linhas duplicadas (mesma estação, início, evento e duração)")
dup = par.duplicated(subset=["Estação", "Data inicial", "Evento", "dur_min"]).sum()
print("Duplicadas:", dup)

print("\n")
print("Durações estranhas")
print(par["dur_min"].describe())
print("Zero:", (par["dur_min"] == 0).sum())
print("Mais de 8 h:", (par["dur_min"] > 480).sum())

fim_calculado = par["Data inicial"] + pd.to_timedelta(par["dur_min"], unit="min")
diferenca = (par["Data final"] - fim_calculado).abs()
print("Linhas onde não bate (dif > 1 min):", (diferenca > pd.Timedelta(minutes=1)).sum())

print("\n")
par[diferenca > pd.Timedelta(minutes=1)][["Data inicial", "Data final", "dur_min"]].head(10)


In [17]:
n_antes = len(par)
par_limpo = (par
             .drop_duplicates(subset=["Estação", "Data inicial", "Evento", "dur_min"])
             .query("dur_min > 0"))
print("Removidas:", n_antes - len(par_limpo))

par_limpo.to_parquet("../data/processed/paradas_limpo.parquet")

Removidas: 9217
